<a href="https://colab.research.google.com/github/yajima-yasutoshi/ADS2026/blob/main/20260929/%E5%9F%BA%E7%A4%8E%E9%9B%86%E8%A8%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 基礎集計：Pandasによるデータ理解・クレンジング・集計


#実行環境とライブラリの準備

本資料では、主に次のライブラリを使用する。

| ライブラリ | 用途 |
|---|---|
| pandas | 表形式データの読み込み、加工、集計 |
| numpy | 数値計算 |
| matplotlib, seaborn | グラフ作成 |
| japanize_matplotlib | 日本語を表示するためのライブラリー |
| scipy | 統計計算 |

In [1]:
%%capture
# Matplotlibで日本語を表示するためのライブラリをインストール
!pip install japanize-matplotlib

In [2]:
# 分析に必要なライブラリを読み込む
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
import scipy.stats as st
import seaborn as sns

#日本語表示のためにインポート
import japanize_matplotlib

##import文

Pythonでは、機能をまとめたライブラリやモジュールを `import` 文で読み込む。

```python
import モジュール名
```

長い名前には別名を付けることができる。

```python
import モジュール名 as 短縮名
```

Pandasには `pd`、NumPyには `np` という短縮名を用いることが一般的である。ただし、短縮名はPythonの文法上の必須条件ではなく、慣例である。

#データの読み込み

この資料で主に用いるデータは `customer_data.xlsx` である。次のいずれかの方法で読み込む。

- **方法A**：GitHub上(インターネット)のファイルをURLから直接読み込む。
- **方法B**：Google Driveに保存したファイルを読み込む。
- **方法C**：Colabへ一時的にアップロードして読み込む。

また、クレンジングや加工を直接元データに上書きすると、処理前後の比較や再検証が困難になる。そのため、読み込んだ直後のデータは、例えば `data_raw` として保持し、分析用データを `data` または `data_clean` として複製し操作するとよい。


##方法A：URLから読み込む

次のコードは、GitHub上のExcelファイルを直接読み込む例である。顧客IDは識別子であるため、文字列として読み込む。

In [3]:
DATA_URL = (
    'https://raw.githubusercontent.com/'
    'yajima-yasutoshi/shunan-u/main/data/customer_data.xlsx'
)
data_raw = pd.read_excel(DATA_URL)
print('読み込み完了:', data_raw.shape)

読み込み完了: (500, 9)


なお、
`pd.read_excel` では、デフォルトで先頭の行の内容を列名として扱う。

##方法B：Google Driveから読み込む

Google Driveの保存したファイルを利用する場合は、以下のセルを使用する。
なお、
`customer_data.xlsx` は Google Drive の`データ`フォルダーに保存されているとする。



In [4]:
#Google ColabでGoogle Driveを使用する場合のみ実行する
from google.colab import drive
drive.mount('/content/drive')

FILE_PATH = '/content/drive/MyDrive/データ/customer_data.xlsx'
data_raw = pd.read_excel(FILE_PATH)
print('読み込み完了:', data_raw.shape)

Mounted at /content/drive
読み込み完了: (500, 9)


##方法C：ColabへPCからアップロードして読み込む

まず、以下のリンク

https://github.com/yajima-yasutoshi/shunan-u/tree/main/data

から、`customer_data.xlsx` を選択し手元のPCの適当な場所に保存する。


その後、以下のセルを実行し、
PC上に保存した `customer_data.xlsx` をアップロードする。

なお、アップロードしたファイルは、Colabのセッション終了時に削除される。
継続利用する場合はGoogle Driveを使用する方が適切である。

In [5]:
# Google Colabへファイルをアップロードする
from google.colab import files
uploaded = files.upload()
file_name = next(iter(uploaded))
data_raw = pd.read_excel(file_name)
print('読み込み完了:', data_raw.shape)

Saving customer_data.xlsx to customer_data.xlsx
読み込み完了: (500, 9)


Colab 環境の左端のメニューからフォルダーのマークをクリックし、ファイルが保存されていることを確認すること。

##読み込み結果を分析用データフレームへ複製する

元データは `data_raw` に保持した。以降では、`data_raw` はそのままにして、 `data` に複製をして使用することにする。

なお、
`data.columns`には列名が格納されている。
列名の前後に不要な空白を取り除く処理行い、以降の扱いを簡単にしている。


In [6]:
data = data_raw.copy() #複製するために .copy()を用いる
data.columns = data.columns.str.strip()

data.head()

,顧客ID,性別,年齢,職業,年収,スマホの所有,スマホ利用時間,Aの利用回数,Bの利用回数
0,81336944,男性,NaN,会社員,393,はい,1.67,1.0,12
1,41990933,女性,51.0,NaN,450,はい,2.44,0.0,11
2,84494791,男性,45.0,会社員,497,はい,0.69,1.0,21
3,29406622,女性,27.0,公務員,313,はい,1.40,NaN,8
4,20205360,女性,61.0,会社員,292,はい,1.77,2.0,24


#データの構造と定義の確認

データ分析の最初の作業は、データの構造、変数の意味、データ型を確認することである。
内容を確認せずに集計すると、単位を誤る、文字列を数値として扱うなどの誤りが生じる。

##データ定義

本資料で扱うデータでは、各列の想定定義は次のとおりである。
実務では、データ辞書や項目定義書で定義を確認する必要がある。

| 列名 | 分析上の型 | 説明 | 単位・注意点 |
|---|---|---|---|
| 顧客ID | 識別子 | 顧客ごとに割り当てられたID | 数値計算の対象外 |
| 性別 | カテゴリ | 顧客の性別 | 値の定義を確認する |
| 年齢 | 数値 | 顧客の年齢 | 歳 |
| 職業 | カテゴリ | 顧客の職業 | 表記揺れに注意する |
| 年収 | 数値 | 顧客の年収 | 万円 |
| スマホの所有 | カテゴリ | スマートフォンの所有状態 | 値の定義を確認する |
| スマホ利用時間 | 数値 | 1日当たりの平均利用時間 | 時間／日 |
| Aの利用回数 | 数値 | サービスAの利用回数 | 集計期間を確認する |
| Bの利用回数 | 数値 | サービスBの利用回数 | 集計期間を確認する |

「Aの利用回数」「Bの利用回数」は、回数の合計をした期間が明確でない。実務分析では、データ辞書などを参照して集計期間を確認する必要がある。

##行数・列数・列名

In [7]:
print('行数:', data.shape[0])
print('列数:', data.shape[1])
print('列名:', data.columns.tolist())

行数: 500
列数: 9
列名: ['顧客ID', '性別', '年齢', '職業', '年収', 'スマホの所有', 'スマホ利用時間', 'Aの利用回数', 'Bの利用回数']


In [8]:
data.columns

Index(['顧客ID', '性別', '年齢', '職業', '年収', 'スマホの所有', 'スマホ利用時間', 'Aの利用回数',
       'Bの利用回数'],
      dtype='object')

`shape` は「行数、列数」の順で値を持つタプルである。行数だけを得る場合は `data.shape[0]`、列数だけを得る場合は `data.shape[1]` を使用する。

##先頭・末尾の確認

In [9]:
# 先頭5行を表示する
data.head()

,顧客ID,性別,年齢,職業,年収,スマホの所有,スマホ利用時間,Aの利用回数,Bの利用回数
0,81336944,男性,NaN,会社員,393,はい,1.67,1.0,12
1,41990933,女性,51.0,NaN,450,はい,2.44,0.0,11
2,84494791,男性,45.0,会社員,497,はい,0.69,1.0,21
3,29406622,女性,27.0,公務員,313,はい,1.40,NaN,8
4,20205360,女性,61.0,会社員,292,はい,1.77,2.0,24


In [10]:
# 末尾5行を表示する
data.tail()

,顧客ID,性別,年齢,職業,年収,スマホの所有,スマホ利用時間,Aの利用回数,Bの利用回数
495,45099496,男性,45.0,公務員,411,はい,2.94,6.0,19
496,30507179,男性,18.0,学生,215,はい,2.01,12.0,8
497,30507179,男性,18.0,学生,215,はい,2.01,12.0,8
498,59997136,男性,65.0,公務員,556,いいえ,0.00,2.0,12
499,99854703,NaN,26.0,会社員,290,はい,1.28,9.0,18


表中の `NaN` や `<NA>` は、値が記録されていないことを示す。これらを**欠損値**と呼ぶ。欠損値は空白や「0」と同じ意味ではないことに注意せよ。

##データ型と欠損の概要

In [11]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   顧客ID     500 non-null    int64  
 1   性別       497 non-null    object 
 2   年齢       496 non-null    float64
 3   職業       499 non-null    object 
 4   年収       500 non-null    int64  
 5   スマホの所有   500 non-null    object 
 6   スマホ利用時間  500 non-null    float64
 7   Aの利用回数   499 non-null    float64
 8   Bの利用回数   500 non-null    int64  
dtypes: float64(3), int64(3), object(3)
memory usage: 35.3+ KB


代表的なPandasのデータ型は次のとおりである。

| dtypeの例 | 主な意味 |
|---|---|
| `int64`, `Int64` | 整数 |
| `float64` | 実数 |
| `string`, `object` | 文字列または混在データ |
| `category` | カテゴリ |
| `bool` | 真偽値 |
| `datetime64[ns]` | 日時 |

`object` となった場合は、
文字列、数値と文字列の混在、日付文字列などが含まれる可能性があるため、実際の値を確認する必要がある。

##想定列の確認

分析コードが期待する列が存在するかを確認する。


In [12]:
expected_columns = [
    '顧客ID', '性別', '年齢', '職業', '年収',
    'スマホの所有', 'スマホ利用時間',
    'Aの利用回数', 'Bの利用回数'
]

#集合型を使い、要素の差を確認する
missing_columns = sorted(set(expected_columns) - set(data.columns))
extra_columns = sorted(set(data.columns) - set(expected_columns))

print('不足している列:', missing_columns)
print('想定外の列:', extra_columns)

if missing_columns:
    raise ValueError(f'必要な列が不足している: {missing_columns}')

不足している列: []
想定外の列: []


#データ品質の確認

データ品質の確認では、少なくとも次の観点を扱う。

1. 完全性：必要な値が欠損していないか。
2. 一意性：同一レコードや同一IDが重複していないか。
3. 妥当性：年齢や利用時間が現実的な範囲にあるか。
4. 整合性：項目間に値の整合性はあるか。
5. 一貫性：カテゴリの表記が統一されているか。

##欠損値の件数と割合

欠損値の個数（レコード数）を項目別に集計する。

In [13]:
data.isna().sum()

,0
顧客ID,0
性別,3
年齢,4
職業,1
年収,0
スマホの所有,0
スマホ利用時間,0
Aの利用回数,1
Bの利用回数,0


件数だけでなく、全体に占める割合も確認する。
10件の欠損でも、全体が20件の場合と10万件の場合では影響が異なるためである。

欠損の件数は `.isna().sum()`、割合は`.isna().mean()` で求める。


In [14]:
missing_summary = pd.DataFrame({
    '欠損件数': data.isna().sum(),
    '欠損率(%)': data.isna().mean().mul(100)
})

missing_summary

,欠損件数,欠損率(%)
顧客ID,0,0.0
性別,3,0.6
年齢,4,0.8
職業,1,0.2
年収,0,0.0
スマホの所有,0,0.0
スマホ利用時間,0,0.0
Aの利用回数,1,0.2
Bの利用回数,0,0.0


##重複行

完全に同じ行の存在を確認する。

In [15]:
#完全に重複した行を調べ、最初に出現した行以外（2回目あるいはそれ以降に出現した行）を True とする
data.duplicated(keep='first')

,0
0,False
1,False
2,False
3,False
4,False
...,...
495,False
496,False
497,True
498,False


In [16]:
#keep=False の場合は、重複した行全てを True とする
data.loc[data.duplicated(keep=False)]

,顧客ID,性別,年齢,職業,年収,スマホの所有,スマホ利用時間,Aの利用回数,Bの利用回数
496,30507179,男性,18.0,学生,215,はい,2.01,12.0,8
497,30507179,男性,18.0,学生,215,はい,2.01,12.0,8


完全に重複した行を一括で削除するためには、
`.drop_duplicates()`
とする。

In [17]:
data = data.drop_duplicates()

In [18]:
data.shape

(499, 9)

##カテゴリ値の確認
カテゴリ型の項目では、まずユニーク値と件数を確認する。
「会社員」「会社員 」「会社 員」などの表記ゆれが存在しないことを確認する。
`.value_counts()` を用いる。
`dropna=False` と指定すると、欠損値も集計の対象になる。

In [19]:
categorical_columns = ['性別', '職業', 'スマホの所有']

for column in categorical_columns:
    print(f'\n--- {column} ---')
    print(data[column].value_counts(dropna=False))


--- 性別 ---
性別
男性     349
女性     147
NaN      3
Name: count, dtype: int64

--- 職業 ---
職業
その他    152
会社員    141
公務員    136
学生      69
NaN      1
Name: count, dtype: int64

--- スマホの所有 ---
スマホの所有
はい     433
いいえ     66
Name: count, dtype: int64


##数値項目の範囲確認

`describe()` は数値項目の件数、平均、標準偏差、最小値、四分位数、最大値を表示する。
表示された値が妥当であるかは、業務知識や項目定義に基づいて判断する必要がある。

In [20]:
numeric_columns = [
    '年齢', '年収', 'スマホ利用時間',
    'Aの利用回数', 'Bの利用回数'
]

data[numeric_columns].describe()

,年齢,年収,スマホ利用時間,Aの利用回数,Bの利用回数
count,495.000000,499.000000,499.000000,498.000000,499.000000
mean,38.929293,428.803607,1.565251,4.845382,14.873747
std,17.789623,251.855024,0.962551,3.827544,4.604661
min,10.000000,0.000000,0.000000,0.000000,4.000000
25%,23.000000,244.000000,1.070000,2.000000,12.000000
50%,39.000000,418.000000,1.530000,4.000000,15.000000
75%,52.500000,605.500000,2.070000,7.000000,18.000000
max,130.000000,1304.000000,6.870000,20.000000,31.000000


なお、データを小さい順に並べたとき、25%地点を第一四分位数、50%地点を第二四分位数、75%地点を第三四分位数と呼ぶ。
したがって、第二四分位数が中央値となる。

例えば、次の9個の値の中央値は20である。

`1, 2, 5, 10, 20, 49, 100, 200, 210`

このデータの平均は約66.3である。極端に大きい値が存在するため、平均は中央値より大きくなる。分布が左右非対称である場合や外れ値がある場合は、平均と中央値双方を確認することが重要である。

##業務ルールによる妥当性確認

実務では、データ定義と業務ルールに基づき、
各数値の妥当な範囲（閾値）が設定される。
次の範囲は例示である。

- 年齢：0以上100以下
- 年収：0以上
- スマホ利用時間：0以上24以下
- サービス利用回数：0以上

In [21]:
quality_flags = pd.DataFrame(index=data.index)
# quality_flags = pd.DataFrame()
quality_flags['年齢範囲外'] = ~data['年齢'].between(0, 100) & data['年齢'].notna()
quality_flags['年収負値'] = (data['年収'] < 0) & data['年収'].notna()
quality_flags['スマホ時間範囲外'] = (
    ~data['スマホ利用時間'].between(0, 24)
    & data['スマホ利用時間'].notna()
)
quality_flags['A利用回数負値'] = (data['Aの利用回数'] < 0) & data['Aの利用回数'].notna()
quality_flags['B利用回数負値'] = (data['Bの利用回数'] < 0) & data['Bの利用回数'].notna()

quality_flags.sum().to_frame('該当件数')

,該当件数
年齢範囲外,3
年収負値,0
スマホ時間範囲外,0
A利用回数負値,0
B利用回数負値,0


##項目間の整合性確認

例えば、スマートフォンを所有していない顧客の利用時間が正の値であれば、定義上の矛盾である可能性がある。共有端末や過去の所有状況など、業務上の説明が存在する可能性もあるため、直ちに誤りと断定でるわけではない。

In [22]:
smartphone_inconsistent = data.loc[
    data['スマホの所有'].isin(['いいえ'])
    & data['スマホ利用時間'].gt(0)
]

print('スマホ所有状態と利用時間が矛盾する可能性のある件数:', len(smartphone_inconsistent))
smartphone_inconsistent.head()

スマホ所有状態と利用時間が矛盾する可能性のある件数: 0


,顧客ID,性別,年齢,職業,年収,スマホの所有,スマホ利用時間,Aの利用回数,Bの利用回数


#データクレンジング

データクレンジングは、単に欠損値や外れ値を削除する作業ではない。分析目的に照らして、どの値を残し、どの値を修正・補完・除外するかを決める作業である。

本資料では、次の方針を採用する。

1. 元データを保持する。
2. 完全重複行は削除候補とする。
3. 文字列では前後の空白を除去する。
4. カテゴリ欠損は、目的に応じて「不明」として保持することを検討する。
5. 数値欠損は一律に0で補完しない。
6. 外れ値は削除前にフラグ化し、原因を確認する。

##分析用データの作成

In [23]:
data_clean = data.copy()

# 文字列列の前後空白を除去する
for column in categorical_columns:
    data_clean[column] = data_clean[column].astype('string').str.strip()

# 完全に同じ行のみを削除する
rows_before = len(data_clean)
data_clean = data_clean.drop_duplicates().copy()
rows_after = len(data_clean)

print('重複削除前:', rows_before)
print('重複削除後:', rows_after)
print('削除件数:', rows_before - rows_after)

重複削除前: 499
重複削除後: 499
削除件数: 0


##欠損値の扱い

欠損値の主な扱いは次のとおりである。

| 方法 | 適する状況 | 主な注意点 |
|---|---|---|
| 削除 | 欠損が少なく、欠損が偶然に生じたと考えられる場合 | 標本数が減り、偏りが生じる可能性がある |
| 固定値で補完 | 「不明」「該当なし」など意味のあるカテゴリを追加する場合 | 0と欠損を混同してはならない |
| 平均・中央値で補完 | 簡易的な前処理を行う場合 | 分散が小さくなり、相関が歪む可能性がある |
| グループ別補完 | グループごとの差を考慮する場合 | グループ定義が妥当である必要がある |
| 欠損のまま扱う | 集計関数が欠損を除外できる場合 | 集計ごとに有効件数を確認する |


欠損値の扱いは、項目の意味や補完後の影響を良く考慮する必要がある。例えば、年収の欠損を0で補完すると、「収入がない」と「不明」を混同するため、通常は適切ではない。

In [24]:
# カテゴリ列の欠損は「不明」として保持する例である
data_clean['性別'] = data_clean['性別'].fillna('不明')
data_clean['職業'] = data_clean['職業'].fillna('不明')
data_clean['スマホの所有'] = data_clean['スマホの所有'].fillna('不明')

# 数値欠損は現時点では補完せず、そのまま保持する
missing_summary_clean = pd.DataFrame({
    '欠損件数': data_clean.isna().sum(),
    '欠損率(%)': data_clean.isna().mean().mul(100)
})

missing_summary_clean.sort_values('欠損率(%)', ascending=False)

,欠損件数,欠損率(%)
年齢,4,0.801603
Aの利用回数,1,0.200401
顧客ID,0,0.000000
職業,0,0.000000
性別,0,0.000000
年収,0,0.000000
スマホの所有,0,0.000000
スマホ利用時間,0,0.000000
Bの利用回数,0,0.000000


### 中央値で補完する例

補完前の値を失わないように補完フラグを作成し、新しい列へ結果を保存する。

In [25]:
data_clean['スマホ利用時間_補完済'] = data_clean['スマホ利用時間']
data_clean['スマホ利用時間_補完フラグ'] = data_clean['スマホ利用時間'].isna()

usage_median = data_clean['スマホ利用時間'].median()
data_clean['スマホ利用時間_補完済'] = (
    data_clean['スマホ利用時間_補完済'].fillna(usage_median)
)

print('補完値（中央値）:', usage_median)
print('補完件数:', data_clean['スマホ利用時間_補完フラグ'].sum())

補完値（中央値）: 1.53
補完件数: 0


##外れ値の確認

外れ値には、次のような異なる原因がある。

- 入力ミスや単位誤り
- 測定機器の異常
- データ連携の不具合
- 実際に存在する希少な顧客
- 母集団が複数の集団から構成されていること

したがって、統計的に外れているという理由だけで削除してはならない。まずフラグを付け、業務上の妥当性を確認する。

### IQR(Interquartile range)による外れ値フラグ

四分位範囲を `IQR = 第三四分位数 - 第一四分位数` とする。一般的な確認基準として、次の範囲外を外れ値候補とする方法がある。

- 下限：第一四分位数 − 1.5 × IQR
- 上限：第三四分位数 ＋ 1.5 × IQR

この基準は外れ値を自動削除する規則ではなく、確認対象を抽出するための目安である。

In [26]:
def iqr_bounds(series, multiplier=1.5):
    """欠損を除外してIQR法の下限・上限を返す関数である。"""
    values = series.dropna()
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    return q1 - multiplier * iqr, q3 + multiplier * iqr

age_lower, age_upper = iqr_bounds(data_clean['年齢'])
data_clean['年齢_IQR外れ値候補'] = (
    (data_clean['年齢'] < age_lower)
    | (data_clean['年齢'] > age_upper)
)

print('年齢のIQR下限:', age_lower)
print('年齢のIQR上限:', age_upper)
print('外れ値候補件数:', data_clean['年齢_IQR外れ値候補'].sum())

年齢のIQR下限: -21.25
年齢のIQR上限: 96.75
外れ値候補件数: 3


In [27]:
data_clean.loc[
    data_clean['年齢_IQR外れ値候補'],
    ['顧客ID', '年齢', '職業']
].sort_values('年齢')

,顧客ID,年齢,職業
98,28123070,130.0,会社員
178,52400051,130.0,会社員
260,31453537,130.0,会社員


### 平均±3標準偏差による確認

平均と標準偏差を用いる方法は、分布が概ね左右対称である場合に用いられることが多い。一方、分布が歪んでいる場合や外れ値が平均・標準偏差自体に影響する場合には注意が必要である。

In [28]:
age_mean = data_clean['年齢'].mean()
age_std = data_clean['年齢'].std()

age_3sd_flag = (
    (data_clean['年齢'] < age_mean - 3 * age_std)
    | (data_clean['年齢'] > age_mean + 3 * age_std)
)

print('平均:', age_mean)
print('標準偏差:', age_std)
print('平均±3標準偏差の範囲外:', age_3sd_flag.sum())

平均: 38.92929292929293
標準偏差: 17.789623348788233
平均±3標準偏差の範囲外: 3


#DataFrameの基本操作

この章では、並べ替え、行・列の選択、条件抽出、列名変更、派生変数作成を扱う。以降の例では `data_clean` を使用する。

##並べ替え

In [29]:
# 年齢の昇順に並べる
data_clean.sort_values('年齢').head(10)

,顧客ID,性別,年齢,職業,年収,スマホの所有,スマホ利用時間,Aの利用回数,Bの利用回数,スマホ利用時間_補完済,スマホ利用時間_補完フラグ,年齢_IQR外れ値候補
308,48345111,女性,10.0,学生,0,いいえ,0.0,3.0,10,0.0,False,False
266,79332527,男性,10.0,学生,0,いいえ,0.0,5.0,20,0.0,False,False
418,38473584,女性,10.0,学生,0,いいえ,0.0,5.0,10,0.0,False,False
482,15287384,男性,10.0,学生,0,いいえ,0.0,10.0,16,0.0,False,False
195,13596379,男性,10.0,学生,0,いいえ,0.0,12.0,11,0.0,False,False
110,91981647,男性,11.0,学生,0,いいえ,0.0,6.0,16,0.0,False,False
303,50985696,男性,11.0,学生,0,いいえ,0.0,8.0,11,0.0,False,False
338,10180519,男性,11.0,学生,0,いいえ,0.0,10.0,14,0.0,False,False
215,96586237,男性,11.0,学生,0,いいえ,0.0,7.0,22,0.0,False,False
177,23544817,女性,11.0,学生,0,いいえ,0.0,3.0,13,0.0,False,False


In [30]:
# 年齢の降順に並べる
data_clean.sort_values('年齢', ascending=False).head(10)

,顧客ID,性別,年齢,職業,年収,スマホの所有,スマホ利用時間,Aの利用回数,Bの利用回数,スマホ利用時間_補完済,スマホ利用時間_補完フラグ,年齢_IQR外れ値候補
98,28123070,男性,130.0,会社員,613,はい,2.05,3.0,15,2.05,False,True
178,52400051,男性,130.0,会社員,457,はい,1.76,2.0,8,1.76,False,True
260,31453537,男性,130.0,会社員,662,はい,3.22,5.0,20,3.22,False,True
54,84790635,女性,65.0,公務員,795,はい,0.83,3.0,13,0.83,False,False
498,59997136,男性,65.0,公務員,556,いいえ,0.00,2.0,12,0.00,False,False
282,37981616,男性,65.0,その他,736,はい,1.52,1.0,16,1.52,False,False
174,17196772,男性,65.0,その他,1238,はい,1.51,2.0,25,1.51,False,False
115,87310579,男性,65.0,会社員,433,はい,1.39,3.0,9,1.39,False,False
489,70847018,男性,65.0,会社員,887,はい,1.56,1.0,15,1.56,False,False
487,27325230,男性,65.0,公務員,694,はい,2.84,4.0,13,2.84,False,False


In [31]:
# 年齢は降順、同年齢では顧客IDを昇順に並べる
data_clean.sort_values(
    ['年齢', '顧客ID'],
    ascending=[False, True]
).head(10)

,顧客ID,性別,年齢,職業,年収,スマホの所有,スマホ利用時間,Aの利用回数,Bの利用回数,スマホ利用時間_補完済,スマホ利用時間_補完フラグ,年齢_IQR外れ値候補
98,28123070,男性,130.0,会社員,613,はい,2.05,3.0,15,2.05,False,True
260,31453537,男性,130.0,会社員,662,はい,3.22,5.0,20,3.22,False,True
178,52400051,男性,130.0,会社員,457,はい,1.76,2.0,8,1.76,False,True
174,17196772,男性,65.0,その他,1238,はい,1.51,2.0,25,1.51,False,False
310,18733313,男性,65.0,その他,974,はい,1.81,1.0,10,1.81,False,False
220,24025262,男性,65.0,その他,978,はい,0.94,5.0,22,0.94,False,False
487,27325230,男性,65.0,公務員,694,はい,2.84,4.0,13,2.84,False,False
282,37981616,男性,65.0,その他,736,はい,1.52,1.0,16,1.52,False,False
317,40801597,女性,65.0,公務員,801,はい,0.72,3.0,23,0.72,False,False
41,42510662,男性,65.0,公務員,548,はい,1.39,2.0,11,1.39,False,False


`sort_values()` は、初期設定では元のDataFrameを変更せず、並べ替えた新しいDataFrameを返す。
結果を保持する場合は `copy()` を使い別の変数へ代入する。

In [32]:
sorted_data = data_clean.sort_values('年齢', ascending=False).copy()
sorted_data.head()

,顧客ID,性別,年齢,職業,年収,スマホの所有,スマホ利用時間,Aの利用回数,Bの利用回数,スマホ利用時間_補完済,スマホ利用時間_補完フラグ,年齢_IQR外れ値候補
98,28123070,男性,130.0,会社員,613,はい,2.05,3.0,15,2.05,False,True
178,52400051,男性,130.0,会社員,457,はい,1.76,2.0,8,1.76,False,True
260,31453537,男性,130.0,会社員,662,はい,3.22,5.0,20,3.22,False,True
54,84790635,女性,65.0,公務員,795,はい,0.83,3.0,13,0.83,False,False
498,59997136,男性,65.0,公務員,556,いいえ,0.00,2.0,12,0.00,False,False


##行番号による選択

位置で選択する場合は `iloc` を使用する。開始位置は含み、終了位置は含まない。

In [33]:
# 21行目と22行目を位置で選択する
data_clean.iloc[20:22]

,顧客ID,性別,年齢,職業,年収,スマホの所有,スマホ利用時間,Aの利用回数,Bの利用回数,スマホ利用時間_補完済,スマホ利用時間_補完フラグ,年齢_IQR外れ値候補
20,86505208,男性,39.0,その他,546,いいえ,0.00,0.0,10,0.00,False,False
21,28562744,男性,45.0,会社員,664,はい,1.01,3.0,12,1.01,False,False


In [34]:
# 先頭3行、3列目から5列目までを選択する
data_clean.iloc[0:3, 2:5]

,年齢,職業,年収
0,NaN,会社員,393
1,51.0,不明,450
2,45.0,会社員,497


##列名による選択

`loc` は行ラベルと列名で選択する。列の選択には、列名のリストを用いる。

In [35]:
selected_data = data_clean.loc[:, ['顧客ID', '性別', '年齢']].copy()
selected_data.head()

,顧客ID,性別,年齢
0,81336944,男性,NaN
1,41990933,女性,51.0
2,84494791,男性,45.0
3,29406622,女性,27.0
4,20205360,女性,61.0


##条件による行の選択

In [36]:
# 年齢が30歳未満の行を選択する
data_clean[data_clean['年齢'] < 30].head()
# .loc を使っても良い
#data_clean.loc[data_clean['年齢'] < 30].head()

,顧客ID,性別,年齢,職業,年収,スマホの所有,スマホ利用時間,Aの利用回数,Bの利用回数,スマホ利用時間_補完済,スマホ利用時間_補完フラグ,年齢_IQR外れ値候補
3,29406622,女性,27.0,公務員,313,はい,1.40,NaN,8,1.40,False,False
6,46119309,女性,12.0,学生,0,いいえ,0.00,4.0,8,0.00,False,False
7,81075284,男性,12.0,学生,0,いいえ,0.00,11.0,13,0.00,False,False
8,39537381,男性,16.0,その他,214,はい,1.14,13.0,19,1.14,False,False
10,43541904,男性,27.0,公務員,413,はい,2.17,14.0,15,2.17,False,False


In [37]:
# 職業が会社員の行を選択する
data_clean[data_clean['職業'] == '会社員'].head()
# data_clean.loc[data_clean['職業'] == '会社員'].head()

,顧客ID,性別,年齢,職業,年収,スマホの所有,スマホ利用時間,Aの利用回数,Bの利用回数,スマホ利用時間_補完済,スマホ利用時間_補完フラグ,年齢_IQR外れ値候補
0,81336944,男性,NaN,会社員,393,はい,1.67,1.0,12,1.67,False,False
2,84494791,男性,45.0,会社員,497,はい,0.69,1.0,21,0.69,False,False
4,20205360,女性,61.0,会社員,292,はい,1.77,2.0,24,1.77,False,False
12,78990911,女性,22.0,会社員,184,はい,1.21,4.0,5,1.21,False,False
14,10309848,女性,62.0,会社員,590,はい,2.16,7.0,31,2.16,False,False


In [38]:
# AND条件：女性かつ年収400万円以上
data_clean[
    (data_clean['性別'] == '女性')
    & (data_clean['年収'] >= 400)
].head()

,顧客ID,性別,年齢,職業,年収,スマホの所有,スマホ利用時間,Aの利用回数,Bの利用回数,スマホ利用時間_補完済,スマホ利用時間_補完フラグ,年齢_IQR外れ値候補
1,41990933,女性,51.0,不明,450,はい,2.44,0.0,11,2.44,False,False
14,10309848,女性,62.0,会社員,590,はい,2.16,7.0,31,2.16,False,False
18,48297048,女性,32.0,その他,401,はい,2.87,2.0,8,2.87,False,False
32,23337424,女性,49.0,公務員,456,いいえ,0.00,2.0,18,0.00,False,False
34,59612559,女性,50.0,公務員,699,はい,1.16,6.0,20,1.16,False,False


In [39]:
# OR条件：会社員または公務員
data_clean[
    data_clean['職業'].isin(['会社員', '公務員'])
].head()

,顧客ID,性別,年齢,職業,年収,スマホの所有,スマホ利用時間,Aの利用回数,Bの利用回数,スマホ利用時間_補完済,スマホ利用時間_補完フラグ,年齢_IQR外れ値候補
0,81336944,男性,NaN,会社員,393,はい,1.67,1.0,12,1.67,False,False
2,84494791,男性,45.0,会社員,497,はい,0.69,1.0,21,0.69,False,False
3,29406622,女性,27.0,公務員,313,はい,1.40,NaN,8,1.40,False,False
4,20205360,女性,61.0,会社員,292,はい,1.77,2.0,24,1.77,False,False
5,55928989,男性,39.0,公務員,317,はい,1.60,1.0,11,1.60,False,False


複数条件では、各条件を丸括弧で囲み、ANDには `&`、ORには `|` を使用する。候補値が複数ある場合は `isin()` が読みやすい。

##列名の変更

`rename()` は、初期設定では元のDataFrameを変更しない。変更結果を変数へ代入する。

In [40]:
data_renamed = data_clean.rename(columns={
    '年収': '年収_万円',
    'Aの利用回数': 'A回数',
    'Bの利用回数': 'B回数'
})

data_renamed.head()

,顧客ID,性別,年齢,職業,年収_万円,スマホの所有,スマホ利用時間,A回数,B回数,スマホ利用時間_補完済,スマホ利用時間_補完フラグ,年齢_IQR外れ値候補
0,81336944,男性,NaN,会社員,393,はい,1.67,1.0,12,1.67,False,False
1,41990933,女性,51.0,不明,450,はい,2.44,0.0,11,2.44,False,False
2,84494791,男性,45.0,会社員,497,はい,0.69,1.0,21,0.69,False,False
3,29406622,女性,27.0,公務員,313,はい,1.40,NaN,8,1.40,False,False
4,20205360,女性,61.0,会社員,292,はい,1.77,2.0,24,1.77,False,False


##特徴量の作成

既存の列から分析に必要な新しい列を作成することを、特徴量の作成と呼ぶ。ここでは、サービスAとBの合計利用回数、およびAの利用比率を作成する。

In [41]:
data_clean['総利用回数'] = (
    data_clean['Aの利用回数']
    + data_clean['Bの利用回数']
)

# 総利用回数が0の場合は0除算を避けるため欠損とする
data_clean['A利用比率'] = np.where(
    data_clean['総利用回数'] > 0,
    data_clean['Aの利用回数'] / data_clean['総利用回数'],
    np.nan
)

data_clean[['Aの利用回数', 'Bの利用回数', '総利用回数', 'A利用比率']].head()

,Aの利用回数,Bの利用回数,総利用回数,A利用比率
0,1.0,12,13.0,0.076923
1,0.0,11,11.0,0.000000
2,1.0,21,22.0,0.045455
3,NaN,8,NaN,NaN
4,2.0,24,26.0,0.076923


##年齢階級の作成

連続値を区間に分ける場合は `pd.cut()` を使用する。区切り方によって集計結果が変わるため、何らかの根拠を持って実行する必要がある。

In [42]:
age_bins = [0, 19, 29, 39, 49, 59, 69, 120]
age_labels = ['19歳以下', '20代', '30代', '40代', '50代', '60代', '70歳以上']

data_clean['年齢階級'] = pd.cut(
    data_clean['年齢'],
    bins=age_bins,
    right=True, # 区間の右を含む → 〇〇以下となる → 例：29歳以下
    labels=age_labels
)

data_clean[['年齢', '年齢階級']].head(5)

,年齢,年齢階級
0,NaN,NaN
1,51.0,50代
2,45.0,40代
3,27.0,20代
4,61.0,60代


#基礎集計

基礎集計では、数値項目の代表値とばらつき、カテゴリ項目の件数と割合を確認する。平均だけでなく、中央値、標準偏差、四分位数、も確認することが重要である。

##数値項目の要約統計量

In [43]:
data_clean.describe()

,顧客ID,年齢,年収,スマホ利用時間,Aの利用回数,Bの利用回数,スマホ利用時間_補完済,総利用回数,A利用比率
count,4.990000e+02,495.000000,499.000000,499.000000,498.000000,499.000000,499.000000,498.000000,498.000000
mean,5.435060e+07,38.929293,428.803607,1.565251,4.845382,14.873747,1.565251,19.732932,0.231725
std,2.544235e+07,17.789623,251.855024,0.962551,3.827544,4.604661,0.962551,6.114843,0.139844
min,1.018052e+07,10.000000,0.000000,0.000000,0.000000,4.000000,0.000000,5.000000,0.000000
25%,3.269559e+07,23.000000,244.000000,1.070000,2.000000,12.000000,1.070000,15.000000,0.117647
50%,5.240005e+07,39.000000,418.000000,1.530000,4.000000,15.000000,1.530000,19.000000,0.200000
75%,7.785447e+07,52.500000,605.500000,2.070000,7.000000,18.000000,2.070000,24.000000,0.333333
max,9.985470e+07,130.000000,1304.000000,6.870000,20.000000,31.000000,6.870000,41.000000,0.655172


集約関数を使う方法もある。第一四分位を求めるのであれば以下のようになる。

In [44]:
data_clean['総利用回数'].quantile(0.25)

np.float64(15.0)

主な集計関数は次のとおりである。

| 関数 | 意味 |
|---|---|
| `count()` | 欠損を除いた件数 |
| `mean()` | 平均 |
| `median()` | 中央値 |
| `min()` | 最小値 |
| `max()` | 最大値 |
| `std()` | 標本標準偏差 |
| `var()` | 標本分散 |
| `quantile(0.25)` | 第一四分位数 |
| `quantile(0.75)` | 第三四分位数 |
| `nunique()` | 異なる値の個数 |
| `unique()` | 異なる値の一覧 |

##特定の条件に絞った集計

In [45]:
# 50歳以上の顧客について、年収の中央値を求める
older_customers = data_clean.loc[data_clean['年齢'] >= 50, '年収']
older_customers.median()

630.0

複数の集計関数を`agg()` でまとめて実行できる。

In [46]:
# 50歳以上の顧客について、年収の有効件数・平均・中央値を求める
older_customers = data_clean.loc[data_clean['年齢'] >= 50, '年収']

older_customers.agg(['count', 'mean', 'median'])

,年収
count,158.000000
mean,634.835443
median,630.000000


以下のように書くことも可能

In [47]:
data_clean.loc[data_clean['年齢'] >= 50, '年収'].agg(['count','mean','median'])

,年収
count,158.000000
mean,634.835443
median,630.000000


In [48]:
# 会社員のスマホ利用時間について平均と中央値を求める
employee_usage = data_clean.loc[
    data_clean['職業'] == '会社員','スマホ利用時間'
]

employee_usage.agg(['count', 'mean', 'median', 'std'])

,スマホ利用時間
count,141.000000
mean,1.708369
median,1.530000
std,1.043333


##カテゴリ項目の件数と割合

`value_counts()` はカテゴリごとの件数を返す。`normalize=True` を指定すると割合を返す。件数と割合の両方があると理解しやすい。

In [49]:
def count_and_ratio(series):
    """カテゴリごとの件数と割合を返す関数である。"""
    counts = series.value_counts(dropna=False)
    ratios = series.value_counts(dropna=False, normalize=True).mul(100)
    return pd.DataFrame({
        '件数': counts,
        '割合(%)': ratios
    })

occupation_summary = count_and_ratio(data_clean['職業'])
occupation_summary

,件数,割合(%)
職業,,
その他,152,30.460922
会社員,141,28.256513
公務員,136,27.254509
学生,69,13.827655
不明,1,0.200401


##最頻値

カテゴリデータで最も多く出現する値を最頻値と呼ぶ。最頻値が複数存在する場合、`mode()` はすべての最頻値を返す。

In [50]:
data_clean['職業'].mode()

,職業
0,その他


In [51]:
# 最頻値が複数ある例である
sample_mode = pd.Series(['a', 'b', 'b', 'c', 'c', 'd'])
sample_mode.mode()

,0
0,b
1,c


# 9. グループ別集計とクロス集計

グループ別集計では、集計項目、分類項目、集計方法を明確にする。

- **集計項目**：年齢、年収、利用回数など
- **分類項目**：性別、職業、年齢階級など
- **集計方法**：件数、平均、中央値、標準偏差、最小値、最大値など


##単一項目で分類する

In [52]:
# dropna=False でNAもカテゴリとして集計する
gender_age_summary = (
  data_clean.groupby('性別',dropna=False)['年齢'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).reset_index()
)
gender_age_summary

,性別,count,mean,median,std,min,max
0,不明,3,32.333333,26.0,13.650397,23.0,48.0
1,女性,147,39.306122,41.0,15.796485,10.0,65.0
2,男性,345,38.826087,39.0,18.628682,10.0,130.0


##複数項目で分類する

In [53]:
group_summary = (
    data_clean.groupby(['性別', '職業'], dropna=False)
    .agg(
        顧客数=('顧客ID', 'nunique'),
        年齢平均=('年齢', 'mean'),
        年齢中央値=('年齢', 'median'),
        年収平均=('年収', 'mean'),
        総利用回数平均=('総利用回数', 'mean')
    )
    .reset_index()
)

group_summary

,性別,職業,顧客数,年齢平均,年齢中央値,年収平均,総利用回数平均
0,不明,会社員,1,26.000000,26.0,290.000000,27.000000
1,不明,公務員,2,35.500000,35.5,362.500000,13.000000
2,女性,その他,51,38.627451,38.0,383.803922,17.078431
3,女性,不明,1,51.000000,51.0,450.000000,11.000000
4,女性,会社員,46,44.652174,50.5,415.847826,18.847826
5,女性,公務員,36,42.027778,46.5,377.027778,18.628571
6,女性,学生,13,14.615385,13.0,63.076923,14.384615
7,男性,その他,101,38.850000,38.0,493.752475,21.534653
8,男性,会社員,94,47.462366,47.0,546.202128,18.882979
9,男性,公務員,98,43.875000,45.5,527.775510,19.112245


`agg()` の名前付き集計を用いると、出力列名と集計対象を同時に指定できる。結果を報告資料へ利用する場合に読みやすい形式である。

##ピボットテーブル

`pivot_table()` は、行と列のカテゴリを組み合わせた集計表を作成する。Excelのピボットテーブルに近い機能である。

In [54]:
occupation_gender_income = pd.pivot_table(
    data_clean,
    index='職業',
    columns='性別',
    values='年収',
    aggfunc=['count', 'mean', 'median'],
    margins=True
)

occupation_gender_income

count                           mean                                      \
性別     不明     女性     男性  All          不明          女性          男性         All   
職業                                                                             
その他   NaN   51.0  101.0  152         NaN  383.803922  493.752475  456.861842   
不明    NaN    1.0    NaN    1         NaN  450.000000         NaN  450.000000   
会社員   1.0   46.0   94.0  141  290.000000  415.847826  546.202128  501.858156   
公務員   2.0   36.0   98.0  136  362.500000  377.027778  527.775510  485.441176   
学生    NaN   13.0   56.0   69         NaN   63.076923  115.678571  105.768116   
All   3.0  147.0  349.0  499  338.333333  364.258503  456.767908  428.803607   

    median                       
性別      不明     女性     男性    All  
職業                               
その他    NaN  320.0  465.0  435.5  
不明     NaN  450.0    NaN  450.0  
会社員  290.0  427.5  545.5  498.0  
公務員  362.5  346.5  514.5  453.0  
学生     NaN    0.0    0.0    0.0  
All  290.0  330.0  449.0  418.0

##クロス集計

カテゴリ同士の組み合わせ件数を確認する場合は `pd.crosstab()` を使用する。件数だけでなく、行割合や列割合を確認すると構成の違いを把握しやすい。

In [55]:
# 件数のクロス集計
occupation_gender_count = pd.crosstab(
    data_clean['職業'],
    data_clean['性別'],
    # margins=True
)

occupation_gender_count

性別,不明,女性,男性
職業,,,
その他,0,51,101
不明,0,1,0
会社員,1,46,94
公務員,2,36,98
学生,0,13,56


In [56]:
# 各職業内での性別構成比（行割合）
occupation_gender_row_ratio = pd.crosstab(
    data_clean['職業'],
    data_clean['性別'],
    normalize='index' # 行で100%となるように集計する 'columns' で各列で100%となる集計となる
).mul(100)

occupation_gender_row_ratio

性別,不明,女性,男性
職業,,,
その他,0.000000,33.552632,66.447368
不明,0.000000,100.000000,0.000000
会社員,0.709220,32.624113,66.666667
公務員,1.470588,26.470588,72.058824
学生,0.000000,18.840580,81.159420


##年齢階級別の利用状況

In [57]:
age_group_usage = (
    data_clean.groupby('年齢階級', observed=False)
    .agg(
        顧客数=('顧客ID', 'nunique'),
        A利用回数平均=('Aの利用回数', 'mean'),
        B利用回数平均=('Bの利用回数', 'mean'),
        総利用回数平均=('総利用回数', 'mean')
    )
    .reset_index()
)

age_group_usage

,年齢階級,顧客数,A利用回数平均,B利用回数平均,総利用回数平均
0,19歳以下,79,8.177215,13.329114,21.506329
1,20代,93,8.108696,13.784946,21.956522
2,30代,78,3.397436,13.269231,16.666667
3,40代,87,3.091954,16.413793,19.505747
4,50代,96,2.968750,16.218750,19.187500
5,60代,59,2.983051,16.389831,19.372881
6,70歳以上,0,NaN,NaN,NaN


グループ間の平均差が見られても、年齢、職業、所得など他の要因が異なる可能性がある。基礎集計は関連を発見するための手段であり、因果関係を証明するものではない。

---
# 参考：よく使用するDataFrame操作

| 目的 | 主な記述 |
|---|---|
| 先頭を表示 | `data.head()` |
| 末尾を表示 | `data.tail()` |
| 行数・列数 | `data.shape` |
| データ型 | `data.dtypes` |
| 概要 | `data.info()` |
| 欠損件数 | `data.isna().sum()` |
| 重複件数 | `data.duplicated().sum()` |
| 列の選択 | `data[['列A', '列B']]` |
| 条件抽出 | `data.loc[条件]` |
| 位置による抽出 | `data.iloc[行, 列]` |
| 並べ替え | `data.sort_values('列名')` |
| 欠損行削除 | `data.dropna(subset=[...])` |
| 重複行削除 | `data.drop_duplicates()` |
| 値の件数 | `data['列'].value_counts()` |
| グループ集計 | `data.groupby('分類列')['数値列'].mean()` |
| クロス集計 | `pd.crosstab(data['列A'], data['列B'])` |
| ピボット集計 | `pd.pivot_table(...)` |